# CubeAnalysis : toutes les figures

Renseigne les huit cubes FITS dans la cellule **Fichiers**, puis exécute tout (*Run All*). Chaque figure s'affiche sous sa cellule et est enregistrée dans `OUTPUT_DIR`.

## Fichiers

In [ ]:
FILES = Dict(
    "density"     => "/chemin/vers/density.fits",      # n [cm^-3]
    "temperature" => "/chemin/vers/temperature.fits",  # T [K]
    "Bx"          => "/chemin/vers/Bx.fits",           # [mG]
    "By"          => "/chemin/vers/By.fits",
    "Bz"          => "/chemin/vers/Bz.fits",
    "Vx"          => "/chemin/vers/Vx.fits",           # [km/s]
    "Vy"          => "/chemin/vers/Vy.fits",
    "Vz"          => "/chemin/vers/Vz.fits",
)

BOX_SIZE   = 50.0                                   # taille de la boîte [pc]
OUTPUT_DIR = joinpath(pwd(), "cubeanalysis_figures")
REPO_PATH  = ""                                     # dossier du dépôt CubeAnalysis ; vide = détection automatique

## Initialisation

In [ ]:
using Pkg

is_cubeanalysis(dir) = (p = joinpath(dir, "Project.toml"); isfile(p) &&
    get(Pkg.TOML.parsefile(p), "name", "") == "CubeAnalysis")

function find_cubeanalysis()
    candidates = String[]
    isempty(REPO_PATH) || push!(candidates, expanduser(REPO_PATH))
    dir = abspath(pwd())
    while true
        push!(candidates, dir, joinpath(dir, "CubeAnalysis"))
        parent = dirname(dir); parent == dir && break; dir = parent
    end
    for base in (homedir(), joinpath(homedir(), "Desktop"), joinpath(homedir(), "Documents"),
                 joinpath(homedir(), ".julia", "dev"))
        push!(candidates, joinpath(base, "CubeAnalysis"))
    end
    for c in candidates
        isdir(c) && is_cubeanalysis(c) && return normpath(c)
    end
    error("Dépôt CubeAnalysis introuvable : renseigne REPO_PATH.")
end

repo = find_cubeanalysis()
Pkg.activate(repo)
Pkg.instantiate()
using CubeAnalysis
CA = CubeAnalysis

# Configuration par défaut du dépôt, pointée sur tes fichiers
for (name, path) in FILES
    isfile(path) || error("Fichier introuvable pour $name : $path")
end
cfg = CA.load_config(joinpath(repo, "config", "cube_analysis.toml"))
for (name, path) in FILES
    cfg["fields"][name]["file"] = abspath(path)
end
cfg["run"]["input_dir"]  = dirname(abspath(FILES["density"]))
cfg["run"]["output_dir"] = abspath(OUTPUT_DIR)
cfg["run"]["formats"]    = ["png"]
cfg["grid"]["box_size"]  = [BOX_SIZE, BOX_SIZE, BOX_SIZE]
mkpath(OUTPUT_DIR)

fields, metadata = CA.load_fields(cfg)
out, fmt, nstride = abspath(OUTPUT_DIR), ["png"], Int(cfg["run"]["sample_stride"])

# Exécute une étape et affiche ses PNG ; une étape impossible est signalée sans tout arrêter
struct PNGFile; path::String; end
Base.show(io::IO, ::MIME"image/png", p::PNGFile) = write(io, read(p.path))

function show_stage(f)
    files = String[]
    try
        f(files)
    catch err
        @warn "Étape ignorée" exception=err
    end
    foreach(p -> display(PNGFile(p)), filter(endswith(".png"), files))
end

@info "Prêt" repo size=CA.analysis_field_size(fields, "density") threads=Threads.nthreads()

## Coupes

In [ ]:
show_stage(f -> CA.plot_slices!(f, fields, metadata, cfg, out, fmt, true))

## Projections et densité de colonne

In [ ]:
show_stage(f -> CA.plot_projections!(f, fields, metadata, cfg, out, fmt, true))

## Histogrammes

In [ ]:
show_stage(f -> CA.plot_histograms!(f, fields, metadata, cfg, out, fmt, true, nstride))

## Diagrammes de phase

In [ ]:
show_stage(f -> CA.plot_phase_diagrams!(f, fields, metadata, cfg, out, fmt, true, nstride))

## Relations B(n) et v(n)

In [ ]:
show_stage(f -> CA.plot_relations!(f, fields, metadata, cfg, out, fmt, true))

## Spectres de puissance

In [ ]:
show_stage(f -> CA.plot_spectra!(f, fields, metadata, cfg, out, fmt, true))

## Spectres vectoriels

In [ ]:
show_stage(f -> CA.plot_vector_spectra!(f, fields, metadata, cfg, out, fmt, true))

## Spectre croisé n et |B|

In [ ]:
show_stage(f -> CA.plot_cross_spectra!(f, fields, cfg, out, fmt, true))

## Alignements et ξ

In [ ]:
show_stage(f -> CA.plot_alignments!(f, fields, metadata, cfg, out, fmt, true))

## Table physique

In [ ]:
show_stage(f -> CA.plot_physics_table!(f, fields, cfg, out, fmt, true))

## PDF de densité

In [ ]:
show_stage(f -> CA.plot_density_pdf!(f, fields, cfg, out, fmt, true))

## Bilan des phases

In [ ]:
show_stage(f -> CA.plot_phase_budget!(f, fields, cfg, out, fmt, true))

## PDF de densité de colonne

In [ ]:
show_stage(f -> CA.plot_column_density_pdf!(f, fields, metadata, cfg, out, fmt, true))

## Fonctions de structure et ESS

In [ ]:
show_stage(f -> CA.plot_advanced_diagnostics!(f, fields, metadata, cfg, out, fmt, true))

## Diagnostics par phase

In [ ]:
show_stage(f -> CA.plot_phase_diagnostics!(f, fields, metadata, cfg, out, fmt, true))

## Observations synthétiques

In [ ]:
show_stage(f -> CA.plot_synthetic_observations!(f, fields, metadata, cfg, out, fmt, true))

## Topologie

In [ ]:
show_stage(f -> CA.plot_topology!(f, fields, metadata, cfg, out, fmt, true))

## Spectres anisotropes

In [ ]:
show_stage(f -> CA.plot_anisotropic_spectra!(f, fields, metadata, cfg, out, fmt, true))

## Fonctions de structure directionnelles

In [ ]:
show_stage(f -> CA.plot_directional_structure!(f, fields, metadata, cfg, out, fmt, true))